The new notebook should perform:

Mount Google Drive.
Restore remote_exec.sh and the PEM key.
Check Colab-to-AWS connectivity.
Start HDFS if required.
Check free AWS disk space.
Create an AWS staging directory.
Create the HDFS raw-zone structure.
Transfer the 325 GeoTIFFs to AWS.
Upload them from AWS staging into HDFS.
Compare source and HDFS file counts and sizes.
Run HDFS health checks.
Save an ingestion manifest.

/data/ghsl/raw/india/
├── smod/
│   ├── epoch=1975/
│   ├── epoch=1980/
│   └── ...
│
├── population/
│   ├── epoch=1975/
│   ├── epoch=1980/
│   └── ...
│
├── built_surface/
│   ├── epoch=1975/
│   ├── epoch=1980/
│   └── ...
│
└── building_height/
    └── year=2018/

In [1]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [4]:
# Diagnose the SSH connection through remote_exec.sh

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

result = subprocess.run(
    ["bash", REMOTE_EXEC, "hostname && whoami"],
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print("\nOUTPUT:")
print(result.stdout or "(no output)")

print("\nSSH ERROR:")
print(result.stderr or "(no error message)")

Return code: 255

OUTPUT:
(no output)

SSH ERROR:
ubuntu@54.144.242.216: Permission denied (publickey).



In [5]:
# ============================================================
# CELL 2 - Prepare SSH key, connect to AWS, and check HDFS
# ============================================================

from pathlib import Path
import shutil
import os
import subprocess

DRIVE_ROOT = Path("/content/drive/MyDrive")
REMOTE_EXEC = DRIVE_ROOT / "remote_exec.sh"
LOCAL_KEY = Path("/content/hadoop-hdfs.pem")

# Find the PEM key in My Drive
pem_matches = list(DRIVE_ROOT.glob("*.pem"))

if not pem_matches:
    raise FileNotFoundError(
        "No .pem file was found in the My Drive root folder."
    )

# Prefer hadoop-hdfs.pem if it exists
drive_key = next(
    (path for path in pem_matches if path.name == "hadoop-hdfs.pem"),
    pem_matches[0],
)

print(f"Remote script: {REMOTE_EXEC}")
print(f"Drive SSH key: {drive_key}")
print(f"Local SSH key: {LOCAL_KEY}")

# Copy the key from Drive to Colab's local filesystem
shutil.copy2(drive_key, LOCAL_KEY)

# SSH requires restrictive permissions for private keys
os.chmod(LOCAL_KEY, 0o600)
os.chmod(REMOTE_EXEC, 0o700)

print("\nSSH key copied and permissions configured.")

# Test AWS connection
print("\n" + "=" * 80)
print("AWS CONNECTION TEST")
print("=" * 80)

aws_test = subprocess.run(
    ["bash", str(REMOTE_EXEC), "hostname && whoami"],
    text=True,
)

if aws_test.returncode != 0:
    raise RuntimeError(
        f"AWS connection failed with return code {aws_test.returncode}."
    )

# Check HDFS capacity and DataNode health
print("\n" + "=" * 80)
print("HDFS CAPACITY AND HEALTH")
print("=" * 80)

hdfs_test = subprocess.run(
    ["bash", str(REMOTE_EXEC), "hdfs dfsadmin -report"],
    text=True,
)

if hdfs_test.returncode != 0:
    raise RuntimeError(
        f"HDFS capacity check failed with return code {hdfs_test.returncode}."
    )

Remote script: /content/drive/MyDrive/remote_exec.sh
Drive SSH key: /content/drive/MyDrive/hadoop-hdfs.pem
Local SSH key: /content/hadoop-hdfs.pem

SSH key copied and permissions configured.

AWS CONNECTION TEST

HDFS CAPACITY AND HEALTH


RuntimeError: HDFS capacity check failed with return code 127.

In [6]:
# ============================================================
# CELL 3 - Locate HDFS on AWS and check cluster capacity
# ============================================================

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

remote_command = r'''
HDFS_BIN="$(command -v hdfs 2>/dev/null || true)"

if [ -z "$HDFS_BIN" ]; then
    for candidate in \
        /usr/local/hadoop/bin/hdfs \
        /opt/hadoop/bin/hdfs \
        /home/ubuntu/hadoop/bin/hdfs \
        /home/ubuntu/hadoop-*/bin/hdfs
    do
        if [ -x "$candidate" ]; then
            HDFS_BIN="$candidate"
            break
        fi
    done
fi

if [ -z "$HDFS_BIN" ]; then
    echo "ERROR: HDFS executable was not found on the AWS instance."
    echo
    echo "Searching for hdfs..."
    find /home/ubuntu /usr/local /opt -type f -name hdfs 2>/dev/null | head -20
    exit 1
fi

HADOOP_HOME="$(dirname "$(dirname "$HDFS_BIN")")"

export HADOOP_HOME
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

echo "HDFS executable: $HDFS_BIN"
echo "HADOOP_HOME: $HADOOP_HOME"

echo
echo "================================================================================"
echo "HDFS CAPACITY AND HEALTH"
echo "================================================================================"

"$HDFS_BIN" dfsadmin -report
'''

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"HDFS check failed with return code {result.returncode}."
    )

HDFS executable: /usr/local/hadoop/bin/hdfs
HADOOP_HOME: /usr/local/hadoop

HDFS CAPACITY AND HEALTH

REMOTE STDERR:
report: Call From ip-172-31-30-94/172.31.30.94 to localhost:9000 failed on connection exception: java.net.ConnectException: Connection refused; For more details see:  http://wiki.apache.org/hadoop/ConnectionRefused



RuntimeError: HDFS check failed with return code 255.

In [7]:
# ============================================================
# CELL 4 - Start HDFS and check cluster capacity
# ============================================================

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

remote_command = r'''
set -e

export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

# Load JAVA_HOME if it is already configured
if [ -f "$HOME/.bashrc" ]; then
    source "$HOME/.bashrc" 2>/dev/null || true
fi

# Detect JAVA_HOME if the non-interactive shell did not load it
if [ -z "$JAVA_HOME" ]; then
    JAVA_PATH="$(readlink -f "$(command -v java)")"
    export JAVA_HOME="$(dirname "$(dirname "$JAVA_PATH")")"
fi

echo "HADOOP_HOME: $HADOOP_HOME"
echo "JAVA_HOME:   $JAVA_HOME"

echo
echo "================================================================================"
echo "STARTING HDFS"
echo "================================================================================"

"$HADOOP_HOME/sbin/start-dfs.sh"

# Give the Hadoop processes a few seconds to initialize
sleep 8

echo
echo "================================================================================"
echo "RUNNING JAVA PROCESSES"
echo "================================================================================"

jps

echo
echo "================================================================================"
echo "HDFS SAFE MODE STATUS"
echo "================================================================================"

"$HADOOP_HOME/bin/hdfs" dfsadmin -safemode get

echo
echo "================================================================================"
echo "HDFS CAPACITY AND HEALTH"
echo "================================================================================"

"$HADOOP_HOME/bin/hdfs" dfsadmin -report
'''

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("\nREMOTE STDERR:")
    print(result.stderr)

if result.returncode != 0:
    print(
        "\nHDFS startup or capacity check failed. "
        f"Return code: {result.returncode}"
    )

HADOOP_HOME: /usr/local/hadoop
JAVA_HOME:   /usr/lib/jvm/java-11-openjdk-amd64

STARTING HDFS
Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [ip-172-31-30-94]

RUNNING JAVA PROCESSES
1936 SecondaryNameNode
1733 DataNode
1596 NameNode
2077 Jps

HDFS SAFE MODE STATUS
Safe mode is OFF

HDFS CAPACITY AND HEALTH
Configured Capacity: 19594608640 (18.25 GB)
Present Capacity: 12417654784 (11.56 GB)
DFS Remaining: 12417605632 (11.56 GB)
DFS Used: 49152 (48 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 1
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 1
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

---------------------------------------

In [ ]:
import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

remote_command = r'''
echo "=== BLOCK DEVICES ==="
lsblk -o NAME,SIZE,TYPE,FSTYPE,MOUNTPOINTS

echo
echo "=== ROOT FILESYSTEM ==="
df -Th /

echo
echo "=== ROOT DEVICE ==="
findmnt -no SOURCE,FSTYPE /
'''

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

=== BLOCK DEVICES ===
NAME          SIZE TYPE FSTYPE   MOUNTPOINTS
loop0        28.2M loop squashfs /snap/amazon-ssm-agent/13009
loop1        28.4M loop squashfs /snap/amazon-ssm-agent/13349
loop2          74M loop squashfs /snap/core22/2437
loop3          74M loop squashfs /snap/core22/2411
loop4        49.3M loop squashfs /snap/snapd/26865
nvme0n1        80G disk          
├─nvme0n1p1  18.9G part ext4     /
├─nvme0n1p13 1023M part ext4     /boot
├─nvme0n1p14    4M part          
└─nvme0n1p15  106M part vfat     /boot/efi

=== ROOT FILESYSTEM ===
Filesystem     Type  Size  Used Avail Use% Mounted on
/dev/root      ext4   19G  6.7G   12G  37% /

=== ROOT DEVICE ===
/dev/nvme0n1p1 ext4

REMOTE STDERR:

Return code: 0


In [9]:
# ============================================================
# Extend Ubuntu root partition and ext4 filesystem
# ============================================================

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

remote_command = r'''
set -e

echo "=== BEFORE RESIZE ==="
lsblk -o NAME,SIZE,TYPE,FSTYPE,MOUNTPOINTS
df -h /

echo
echo "=== EXTENDING ROOT PARTITION ==="
sudo growpart /dev/nvme0n1 1

echo
echo "=== EXTENDING EXT4 FILESYSTEM ==="
sudo resize2fs /dev/nvme0n1p1

echo
echo "=== AFTER RESIZE ==="
lsblk -o NAME,SIZE,TYPE,FSTYPE,MOUNTPOINTS
df -h /
'''

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Ubuntu partition expansion failed with return code "
        f"{result.returncode}."
    )

=== BEFORE RESIZE ===
NAME          SIZE TYPE FSTYPE   MOUNTPOINTS
loop0        28.2M loop squashfs /snap/amazon-ssm-agent/13009
loop1        28.4M loop squashfs /snap/amazon-ssm-agent/13349
loop2          74M loop squashfs /snap/core22/2437
loop3          74M loop squashfs /snap/core22/2411
loop4        49.3M loop squashfs /snap/snapd/26865
nvme0n1        80G disk          
├─nvme0n1p1  18.9G part ext4     /
├─nvme0n1p13 1023M part ext4     /boot
├─nvme0n1p14    4M part          
└─nvme0n1p15  106M part vfat     /boot/efi
Filesystem      Size  Used Avail Use% Mounted on
/dev/root        19G  6.7G   12G  37% /

=== EXTENDING ROOT PARTITION ===
CHANGED: partition=1 start=2324480 old: size=39618527 end=41943006 new: size=165447647 end=167772126

=== EXTENDING EXT4 FILESYSTEM ===
Filesystem at /dev/nvme0n1p1 is mounted on /; on-line resizing required
old_desc_blocks = 3, new_desc_blocks = 10
The filesystem on /dev/nvme0n1p1 is now 20680955 (4k) blocks long.


=== AFTER RESIZE ===
NAME    

In [10]:
# ============================================================
# Verify expanded HDFS capacity
# ============================================================

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

remote_command = r'''
export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

echo "================================================================================"
echo "HDFS CAPACITY AFTER EBS EXPANSION"
echo "================================================================================"

hdfs dfsadmin -report

echo
echo "================================================================================"
echo "HDFS FILESYSTEM SPACE"
echo "================================================================================"

hdfs dfs -df -h /
'''

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"HDFS capacity verification failed with return code "
        f"{result.returncode}."
    )

HDFS CAPACITY AFTER EBS EXPANSION
Configured Capacity: 82085269504 (76.45 GB)
Present Capacity: 74908303360 (69.76 GB)
DFS Remaining: 74908254208 (69.76 GB)
DFS Used: 49152 (48 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 1
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 1
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
Live datanodes (1):

Name: 127.0.0.1:9866 (localhost)
Hostname: ip-172-31-30-94.ec2.internal
Decommission Status : Normal
Configured Capacity: 82085269504 (76.45 GB)
DFS Used: 49152 (48 KB)
Non DFS Used: 7160188928 (6.67 GB)
DFS Remaining: 74908254208 (69.76 GB)
DFS Used%: 0.00%
DFS Remaining%: 91.

next cell. It will:

Confirm all source files.
Create the HDFS raw-zone structure.
Transfer one file at a time: Drive → AWS staging → HDFS.
Use replication factor 1.
Verify each HDFS file size.
Delete each temporary AWS copy.
Skip already-correct files if interrupted and rerun.
Save an ingestion log in Google Drive


In [4]:
# ============================================================
# GHSL INDIA: GOOGLE DRIVE -> AWS STAGING -> HDFS RAW ZONE
# Resume-safe, one-file-at-a-time ingestion
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import csv
import os
import re
import shlex
import subprocess
import time

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SOURCE_DIR = Path(
    "/content/drive/MyDrive/GHSL_India_Full_R2023A"
)

CONTROL_DIR = Path(
    "/content/drive/MyDrive/GHSL_India_Export_Control"
)

REMOTE_EXEC = Path(
    "/content/drive/MyDrive/remote_exec.sh"
)

SSH_KEY = Path(
    "/content/hadoop-hdfs.pem"
)

AWS_USER = "ubuntu"
AWS_HOST = "54.144.242.216"

AWS_STAGING_DIR = "/home/ubuntu/ghsl_ingestion_staging"
HDFS_ROOT = "/data/ghsl/raw/india"

EXPECTED_FILE_COUNT = 325
EXPECTED_SIZE_GIB_APPROX = 6.555

LOG_FILE = CONTROL_DIR / "ghsl_hdfs_ingestion_log.csv"

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

if not SOURCE_DIR.is_dir():
    raise FileNotFoundError(
        f"GHSL source folder not found:\n{SOURCE_DIR}"
    )

if not REMOTE_EXEC.is_file():
    raise FileNotFoundError(
        f"Remote execution script not found:\n{REMOTE_EXEC}"
    )

if not SSH_KEY.is_file():
    raise FileNotFoundError(
        f"Local SSH key not found:\n{SSH_KEY}"
    )

os.chmod(SSH_KEY, 0o600)
CONTROL_DIR.mkdir(parents=True, exist_ok=True)

source_files = sorted(
    path
    for path in SOURCE_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in {".tif", ".tiff"}
)

source_count = len(source_files)
source_bytes = sum(path.stat().st_size for path in source_files)
source_gib = source_bytes / (1024 ** 3)

print("=" * 100)
print("GHSL HDFS INGESTION")
print("=" * 100)
print(f"Source folder : {SOURCE_DIR}")
print(f"GeoTIFF files: {source_count}")
print(f"Total size    : {source_gib:.3f} GiB")
print(f"HDFS root     : {HDFS_ROOT}")
print(f"AWS staging   : {AWS_STAGING_DIR}")
print(f"Ingestion log : {LOG_FILE}")

if source_count != EXPECTED_FILE_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_FILE_COUNT} GeoTIFF files, "
        f"but found {source_count}. Ingestion stopped."
    )

if abs(source_gib - EXPECTED_SIZE_GIB_APPROX) > 0.25:
    raise RuntimeError(
        f"Expected approximately {EXPECTED_SIZE_GIB_APPROX:.3f} GiB, "
        f"but found {source_gib:.3f} GiB. Ingestion stopped."
    )

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def run_remote(command, check=True):
    result = subprocess.run(
        ["bash", str(REMOTE_EXEC), command],
        text=True,
        capture_output=True,
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            "Remote command failed.\n"
            f"Command: {command}\n"
            f"Return code: {result.returncode}\n"
            f"STDOUT:\n{result.stdout}\n"
            f"STDERR:\n{result.stderr}"
        )

    return result


def determine_hdfs_directory(filename):
    """
    Convert a GHSL filename into the appropriate partitioned HDFS path.

    Examples:
      population_1975    -> /data/ghsl/raw/india/population/epoch=1975
      built_surface_2020 -> /data/ghsl/raw/india/built_surface/epoch=2020
      smod_2015          -> /data/ghsl/raw/india/smod/epoch=2015
      building_height    -> /data/ghsl/raw/india/building_height/year=2018
    """

    name = filename.lower()

    if "building_height" in name or "building-height" in name:
        year_match = re.search(r"(19|20)\d{2}", name)
        year = year_match.group(0) if year_match else "2018"
        return f"{HDFS_ROOT}/building_height/year={year}"

    if "built_surface" in name or "built-surface" in name:
        product = "built_surface"
    elif "population" in name:
        product = "population"
    elif "smod" in name:
        product = "smod"
    else:
        raise ValueError(
            f"Could not identify GHSL product from filename: {filename}"
        )

    year_match = re.search(
        r"(?:1975|1980|1985|1990|1995|2000|2005|2010|2015|2020)",
        name,
    )

    if not year_match:
        raise ValueError(
            f"Could not identify epoch from filename: {filename}"
        )

    epoch = year_match.group(0)
    return f"{HDFS_ROOT}/{product}/epoch={epoch}"


# ------------------------------------------------------------
# Validate filename classification before transferring anything
# ------------------------------------------------------------

classified_files = []

for source_path in source_files:
    hdfs_directory = determine_hdfs_directory(source_path.name)
    hdfs_path = f"{hdfs_directory}/{source_path.name}"

    classified_files.append(
        {
            "source_path": source_path,
            "source_size": source_path.stat().st_size,
            "hdfs_directory": hdfs_directory,
            "hdfs_path": hdfs_path,
        }
    )

print("\nFilename classification completed successfully.")

# ------------------------------------------------------------
# Prepare AWS staging and HDFS
# ------------------------------------------------------------

prepare_command = f'''
set -e

export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

mkdir -p {shlex.quote(AWS_STAGING_DIR)}

hdfs dfsadmin -safemode leave >/dev/null 2>&1 || true

hdfs dfs -mkdir -p {shlex.quote(HDFS_ROOT)}

echo "HDFS is ready."
hdfs dfs -df -h /
'''

prepare_result = run_remote(prepare_command)

print("\n" + "=" * 100)
print("REMOTE PREPARATION")
print("=" * 100)
print(prepare_result.stdout)

# ------------------------------------------------------------
# Initialize append-only ingestion log
# ------------------------------------------------------------

log_fields = [
    "timestamp_utc",
    "status",
    "filename",
    "source_bytes",
    "hdfs_bytes",
    "hdfs_path",
    "elapsed_seconds",
    "message",
]

if not LOG_FILE.exists():
    with LOG_FILE.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as log_handle:
        writer = csv.DictWriter(log_handle, fieldnames=log_fields)
        writer.writeheader()

# ------------------------------------------------------------
# Ingest one file at a time
# ------------------------------------------------------------

uploaded = 0
skipped = 0
failed = 0
processed_bytes = 0
start_time = time.time()

for index, item in enumerate(classified_files, start=1):

    source_path = item["source_path"]
    source_size = item["source_size"]
    hdfs_directory = item["hdfs_directory"]
    hdfs_path = item["hdfs_path"]

    file_start = time.time()

    print("\n" + "-" * 100)
    print(
        f"[{index:03d}/{source_count}] "
        f"{source_path.name}"
    )
    print(
        f"Source size: {source_size / (1024 ** 2):,.2f} MiB"
    )
    print(f"HDFS path : {hdfs_path}")

    remote_staging_path = (
        f"{AWS_STAGING_DIR}/{source_path.name}"
    )

    try:
        # ----------------------------------------------------
        # Resume check: skip only when HDFS size is exact
        # ----------------------------------------------------

        quoted_hdfs_path = shlex.quote(hdfs_path)

        size_check_command = f'''
export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

if hdfs dfs -test -e {quoted_hdfs_path}; then
    hdfs dfs -stat %b {quoted_hdfs_path}
else
    echo "MISSING"
fi
'''

        size_result = run_remote(size_check_command)
        remote_value = size_result.stdout.strip().splitlines()[-1]

        if (
            remote_value != "MISSING"
            and remote_value.isdigit()
            and int(remote_value) == source_size
        ):
            skipped += 1
            processed_bytes += source_size
            elapsed = time.time() - file_start

            print("Status: SKIPPED, correct file already exists in HDFS.")

            log_row = {
                "timestamp_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "status": "SKIPPED",
                "filename": source_path.name,
                "source_bytes": source_size,
                "hdfs_bytes": int(remote_value),
                "hdfs_path": hdfs_path,
                "elapsed_seconds": f"{elapsed:.2f}",
                "message": "Correct-size file already existed.",
            }

        else:
            # ------------------------------------------------
            # Remove stale staging file
            # ------------------------------------------------

            run_remote(
                f"rm -f {shlex.quote(remote_staging_path)}"
            )

            # ------------------------------------------------
            # Colab/Drive -> AWS staging
            # ------------------------------------------------

            scp_command = [
                "scp",
                "-o", "StrictHostKeyChecking=no",
                "-o", "UserKnownHostsFile=/dev/null",
                "-o", "BatchMode=yes",
                "-o", "ConnectTimeout=30",
                "-i", str(SSH_KEY),
                str(source_path),
                f"{AWS_USER}@{AWS_HOST}:{remote_staging_path}",
            ]

            scp_result = subprocess.run(
                scp_command,
                text=True,
                capture_output=True,
            )

            if scp_result.returncode != 0:
                raise RuntimeError(
                    "SCP transfer failed.\n"
                    f"STDOUT:\n{scp_result.stdout}\n"
                    f"STDERR:\n{scp_result.stderr}"
                )

            # ------------------------------------------------
            # AWS staging -> HDFS
            # Verify both staging and HDFS sizes
            # Delete staging after successful verification
            # ------------------------------------------------

            ingest_command = f'''
set -e

export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

STAGING_FILE={shlex.quote(remote_staging_path)}
HDFS_DIRECTORY={shlex.quote(hdfs_directory)}
HDFS_FILE={quoted_hdfs_path}
EXPECTED_SIZE={source_size}

ACTUAL_STAGING_SIZE="$(stat -c %s "$STAGING_FILE")"

if [ "$ACTUAL_STAGING_SIZE" -ne "$EXPECTED_SIZE" ]; then
    echo "Staging size mismatch."
    echo "Expected: $EXPECTED_SIZE"
    echo "Actual:   $ACTUAL_STAGING_SIZE"
    exit 21
fi

hdfs dfs -mkdir -p "$HDFS_DIRECTORY"

if hdfs dfs -test -e "$HDFS_FILE"; then
    hdfs dfs -rm -f "$HDFS_FILE"
fi

hdfs dfs -Ddfs.replication=1 -put "$STAGING_FILE" "$HDFS_FILE"

HDFS_SIZE="$(hdfs dfs -stat %b "$HDFS_FILE")"

if [ "$HDFS_SIZE" -ne "$EXPECTED_SIZE" ]; then
    echo "HDFS size mismatch."
    echo "Expected: $EXPECTED_SIZE"
    echo "Actual:   $HDFS_SIZE"
    exit 22
fi

hdfs dfs -setrep -w 1 "$HDFS_FILE" >/dev/null

rm -f "$STAGING_FILE"

echo "$HDFS_SIZE"
'''

            ingest_result = run_remote(ingest_command)
            hdfs_size = int(
                ingest_result.stdout.strip().splitlines()[-1]
            )

            uploaded += 1
            processed_bytes += source_size
            elapsed = time.time() - file_start

            print("Status: UPLOADED AND VERIFIED")
            print(
                f"Time  : {elapsed:.1f} seconds"
            )

            log_row = {
                "timestamp_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "status": "UPLOADED",
                "filename": source_path.name,
                "source_bytes": source_size,
                "hdfs_bytes": hdfs_size,
                "hdfs_path": hdfs_path,
                "elapsed_seconds": f"{elapsed:.2f}",
                "message": "Uploaded and size verified.",
            }

        with LOG_FILE.open(
            "a",
            newline="",
            encoding="utf-8",
        ) as log_handle:
            writer = csv.DictWriter(
                log_handle,
                fieldnames=log_fields,
            )
            writer.writerow(log_row)

        total_elapsed = time.time() - start_time
        completion = index / source_count
        estimated_total = (
            total_elapsed / completion if completion else 0
        )
        remaining_seconds = max(
            estimated_total - total_elapsed,
            0,
        )

        print(
            f"Progress: {index}/{source_count} "
            f"({completion * 100:.1f}%)"
        )
        print(
            f"Processed: "
            f"{processed_bytes / (1024 ** 3):.3f} GiB"
        )
        print(
            f"Estimated remaining time: "
            f"{remaining_seconds / 60:.1f} minutes"
        )

    except Exception as error:
        failed += 1
        elapsed = time.time() - file_start

        print("Status: FAILED")
        print(str(error))

        with LOG_FILE.open(
            "a",
            newline="",
            encoding="utf-8",
        ) as log_handle:
            writer = csv.DictWriter(
                log_handle,
                fieldnames=log_fields,
            )
            writer.writerow(
                {
                    "timestamp_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "status": "FAILED",
                    "filename": source_path.name,
                    "source_bytes": source_size,
                    "hdfs_bytes": "",
                    "hdfs_path": hdfs_path,
                    "elapsed_seconds": f"{elapsed:.2f}",
                    "message": str(error).replace("\n", " | "),
                }
            )

        # Remove partial AWS staging file before stopping
        run_remote(
            f"rm -f {shlex.quote(remote_staging_path)}",
            check=False,
        )

        raise

# ------------------------------------------------------------
# Final HDFS verification
# ------------------------------------------------------------

verification_command = f'''
set -e

export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

echo "=== HDFS GEOTIFF COUNT ==="
hdfs dfs -find {shlex.quote(HDFS_ROOT)} -type f \
    | grep -Ei '\\.tiff?$' \
    | wc -l

echo
echo "=== HDFS RAW-ZONE SIZE ==="
hdfs dfs -du -s -h {shlex.quote(HDFS_ROOT)}

echo
echo "=== HDFS REMAINING CAPACITY ==="
hdfs dfs -df -h /

echo
echo "=== HDFS HEALTH ==="
hdfs fsck {shlex.quote(HDFS_ROOT)} \
    -files -blocks -locations \
    | tail -20

echo
echo "=== AWS STAGING REMAINDER ==="
find {shlex.quote(AWS_STAGING_DIR)} \
    -maxdepth 1 -type f | wc -l
'''

verification_result = run_remote(verification_command)

total_minutes = (time.time() - start_time) / 60

print("\n" + "=" * 100)
print("INGESTION COMPLETED")
print("=" * 100)
print(f"Uploaded files : {uploaded}")
print(f"Skipped files  : {skipped}")
print(f"Failed files   : {failed}")
print(f"Total files    : {uploaded + skipped}")
print(f"Elapsed time   : {total_minutes:.1f} minutes")
print(f"Log saved to   : {LOG_FILE}")

print("\n" + verification_result.stdout)

if uploaded + skipped != EXPECTED_FILE_COUNT:
    raise RuntimeError(
        "Final file count does not match the expected "
        f"{EXPECTED_FILE_COUNT} files."
    )

print("=" * 100)
print("FINAL RESULT: GHSL HDFS INGESTION SUCCESSFUL")
print("=" * 100)

GHSL HDFS INGESTION
Source folder : /content/drive/MyDrive/GHSL_India_Full_R2023A
GeoTIFF files: 325
Total size    : 6.555 GiB
HDFS root     : /data/ghsl/raw/india
AWS staging   : /home/ubuntu/ghsl_ingestion_staging
Ingestion log : /content/drive/MyDrive/GHSL_India_Export_Control/ghsl_hdfs_ingestion_log.csv

Filename classification completed successfully.

REMOTE PREPARATION
HDFS is ready.
Filesystem               Size    Used  Available  Use%
hdfs://localhost:9000  76.4 G  92.3 M     69.7 G    0%


----------------------------------------------------------------------------------------------------
[001/325] ghsl_india_building_height_2018_100m_r2023a-0000000000-0000000000.tif
Source size: 55.66 MiB
HDFS path : /data/ghsl/raw/india/building_height/year=2018/ghsl_india_building_height_2018_100m_r2023a-0000000000-0000000000.tif
Status: SKIPPED, correct file already exists in HDFS.
Progress: 1/325 (0.3%)
Processed: 0.054 GiB
Estimated remaining time: 36.2 minutes

------------------------

In [1]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True,
)

print("Google Drive remounted successfully.")

Mounted at /content/drive
Google Drive remounted successfully.


In [2]:
from pathlib import Path
import shutil
import os

DRIVE_KEY = Path("/content/drive/MyDrive/hadoop-hdfs.pem")
LOCAL_KEY = Path("/content/hadoop-hdfs.pem")

if not DRIVE_KEY.is_file():
    raise FileNotFoundError(f"SSH key not found: {DRIVE_KEY}")

shutil.copy2(DRIVE_KEY, LOCAL_KEY)
os.chmod(LOCAL_KEY, 0o600)

print("SSH key restored successfully.")


SSH key restored successfully.


In [3]:
from pathlib import Path

SOURCE_DIR = Path(
    "/content/drive/MyDrive/GHSL_India_Full_R2023A"
)

source_files = sorted(SOURCE_DIR.glob("*.tif"))

print("Source folder accessible:", SOURCE_DIR.is_dir())
print("GeoTIFF files found:", len(source_files))
print(
    "Total size:",
    f"{sum(path.stat().st_size for path in source_files) / (1024 ** 3):.3f} GiB",
)

if len(source_files) != 325:
    raise RuntimeError(
        f"Expected 325 GeoTIFF files, found {len(source_files)}."
    )

Source folder accessible: True
GeoTIFF files found: 325
Total size: 6.555 GiB


In [5]:
# ============================================================
# Correct final verification of GHSL files in HDFS
# ============================================================

import subprocess

REMOTE_EXEC = "/content/drive/MyDrive/remote_exec.sh"

remote_command = r'''
set -e

export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

HDFS_ROOT="/data/ghsl/raw/india"

echo "================================================================================"
echo "HDFS GEOTIFF FILE COUNT"
echo "================================================================================"

FILE_COUNT="$(
    hdfs dfs -ls -R "$HDFS_ROOT" \
    | awk '$1 ~ /^-/ {print $NF}' \
    | grep -Ei '\.tiff?$' \
    | wc -l
)"

echo "$FILE_COUNT"

echo
echo "================================================================================"
echo "HDFS TOTAL FILE COUNT"
echo "================================================================================"

hdfs dfs -ls -R "$HDFS_ROOT" \
    | awk '$1 ~ /^-/ {count++} END {print count+0}'

echo
echo "================================================================================"
echo "FILES BY PRODUCT"
echo "================================================================================"

for product in building_height built_surface population smod
do
    PRODUCT_PATH="$HDFS_ROOT/$product"

    COUNT="$(
        hdfs dfs -ls -R "$PRODUCT_PATH" 2>/dev/null \
        | awk '$1 ~ /^-/ {count++} END {print count+0}'
    )"

    SIZE="$(
        hdfs dfs -du -s -h "$PRODUCT_PATH" 2>/dev/null \
        | awk '{print $1 " " $2}'
    )"

    printf "%-20s files=%-5s size=%s\n" \
        "$product" "$COUNT" "$SIZE"
done

echo
echo "================================================================================"
echo "HDFS RAW-ZONE SIZE"
echo "================================================================================"

hdfs dfs -du -s -h "$HDFS_ROOT"

echo
echo "================================================================================"
echo "HDFS REPLICATION FACTORS"
echo "================================================================================"

hdfs dfs -ls -R "$HDFS_ROOT" \
    | awk '$1 ~ /^-/ {print $2}' \
    | sort \
    | uniq -c

echo
echo "================================================================================"
echo "HDFS HEALTH SUMMARY"
echo "================================================================================"

hdfs fsck "$HDFS_ROOT" \
    | grep -E \
    "Status:|Total size:|Total files:|Total blocks|Missing blocks:|Missing replicas:|Under replicated blocks:"

echo
echo "================================================================================"
echo "HDFS REMAINING CAPACITY"
echo "================================================================================"

hdfs dfs -df -h /
'''

result = subprocess.run(
    ["bash", REMOTE_EXEC, remote_command],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("REMOTE STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Final verification failed with return code "
        f"{result.returncode}."
    )

HDFS GEOTIFF FILE COUNT
325

HDFS TOTAL FILE COUNT
325

FILES BY PRODUCT
building_height      files=15    size=373.6 M
built_surface        files=150   size=1.2 G
population           files=150   size=5.0 G
smod                 files=10    size=11.4 M

HDFS RAW-ZONE SIZE
6.6 G  6.6 G  /data/ghsl/raw/india

HDFS REPLICATION FACTORS
    325 1

HDFS HEALTH SUMMARY
Status: HEALTHY
 Total size:	7038596744 B
 Total files:	325
 Total blocks (validated):	325 (avg. block size 21657220 B)
 Missing blocks:		0
 Missing replicas:		0 (0.0 %)
 Total size:	0 B
 Total files:	0

HDFS REMAINING CAPACITY
Filesystem               Size   Used  Available  Use%
hdfs://localhost:9000  76.4 G  6.6 G     63.1 G    9%

REMOTE STDERR:
Connecting to namenode via http://localhost:9870/fsck?ugi=ubuntu&path=%2Fdata%2Fghsl%2Fraw%2Findia

